In [ ]:
import emoji
import pandas as pd
import re
from collections import defaultdict, Counter

# LLM prompt context

---

WhatsApp Chat Analysis (iOS): Assist with Python tool.
Dependencies: re, collections (Counter,defaultdict),pathlib,emoji
Storage:

List: `messages = [ {...}, {...}, ... ]`
{
  "date": "DD/MM/YYYY",
  "time": "HH.MM.SS",
  "sender": "cleaned name",
  "msg": "full text (may have newlines)"
}

DataFrame:
Columns:
['sender','msg','datetime','year','month','day','hour','minute','weekday','type']
type ∈ ['photo','video','sticker','attachment','text']

---

In [ ]:
import re

line_re = re.compile(
    r"^\[(?P<date>\d{2}/\d{2}/\d{4}), (?P<time>\d{2}\.\d{2}\.\d{2})\] (?P<sender>.*?): (?P<msg>.*)$"
)

messages = []
current = None

with open("/Users/mikaelthorup/_chat.txt", encoding="utf-8") as f:
    f = [line.rstrip("\n") for line in f]
    # Clean Unicode characters
    f = [line.replace("\u200e","").replace("\u2068", "").replace("\u2069", "") for line in f]

    for raw in f:
        line = raw.rstrip("\n")

        m = line_re.match(line)
        if m:
            # starting a new message → push previous
            if current:
                messages.append(current)

            current = {
                "date": m.group("date"),
                "time": m.group("time"),
                "sender": m.group("sender"),
                "msg": m.group("msg"),
            }
        else:
            # continuation → append to last message
            if current:
                current["msg"] += "\n" + line

# add last message
if current:
    messages.append(current)

In [ ]:
# debug
for m in messages[0:10]:
    print(f"{m['date']} {m['time']} | {m['sender']}: {m['msg']}")

In [ ]:
def classify_message(msg_text):
    """Classify message type based on common WhatsApp iOS export patterns."""
    msg_text = msg_text.strip()  # remove whitespace
    msg_text = msg_text.replace("\u200e", "")  # remove left-to-right mark if present
    
    if msg_text.startswith("<attached:"):
        # Extract type from filename if possible
        match = re.search(r"-([A-Z]+)-\d{4}-\d{2}-\d{2}-\d{2}-\d{2}-\d{2}", msg_text)
        if match:
            return match.group(1).lower()  # sticker, photo, video, etc.
        # fallback if filename format unknown
        ext_match = re.search(r"\.(\w+)>$", msg_text)
        if ext_match:
            ext = ext_match.group(1).lower()
            if ext in ("jpg", "jpeg", "png", "gif"):
                return "photo"
            elif ext in ("mp4", "mov", "avi"):
                return "video"
            elif ext in ("webp",):
                return "sticker"
        return "attachment"
    else:
        return "text"

In [ ]:
# Convert messages to DataFrame with derived fields
df = pd.DataFrame(messages)

# Parse datetime
df["datetime"] = pd.to_datetime(df["date"] + " " + df["time"], format="%d/%m/%Y %H.%M.%S")

# Derived features
df["year"] = df["datetime"].dt.year
df["month"] = df["datetime"].dt.month
df["day"] = df["datetime"].dt.day
df["hour"] = df["datetime"].dt.hour
df["minute"] = df["datetime"].dt.minute
df["weekday"] = df["datetime"].dt.weekday  # 0 = Monday

# Message classification
df["type"] = df["msg"].apply(classify_message)

df = df.drop(columns=["date", "time"])

# print(df.head())


In [ ]:
# Count messages per person
len(df)

In [ ]:
import string

# Ensure 'datetime' column is datetime
df['datetime'] = pd.to_datetime(df['datetime'], format='%d/%m/%Y %H.%M.%S')

# Helper function to count emojis in a string
def count_emojis(text):
    return sum(1 for char in text if char in emoji.EMOJI_DATA)

# Final day of the chat
last_day = df['datetime'].max().normalize()

# First message date per sender
first_msg = df.groupby('sender')['datetime'].min().rename('first_message_day')

# Days active per sender
days_active = (last_day - first_msg.dt.normalize()).dt.days + 1
days_active = days_active.rename('days_active')

# Total messages per sender
total_msgs = df.groupby('sender').size().rename('total_messages')

# Count of messages by type per sender
type_counts = df.pivot_table(index='sender', columns='type', aggfunc='size', fill_value=0)

# Count of emojis per sender
emoji_counts = df.groupby('sender')['msg'].apply(lambda x: sum(count_emojis(m) for m in x)).rename('n_emoji')

# Total character count per sender
char_count = df.groupby('sender')['msg'].apply(lambda x: sum(len(m) for m in x)).rename('n_chars')

# Count of newlines per sender
newline_count = df.groupby('sender')['msg'].apply(lambda x: sum(m.count('\n') for m in x)).rename('n_newlines')

# Count of punctuation characters per sender
punct_count = df.groupby('sender')['msg'].apply(lambda x: sum(sum(1 for c in m if c in string.punctuation) for m in x)).rename('n_punct')

# Percent of uppercase letters per sender
def pct_uppercase(msgs):
    total_upper = sum(sum(1 for c in m if c.isupper()) for m in msgs)
    total_letters = sum(sum(1 for c in m if c.isalpha()) for m in msgs)
    return (total_upper / total_letters * 100) if total_letters > 0 else 0

pct_upper = df.groupby('sender')['msg'].apply(pct_uppercase).rename('pct_uppercase')

# Combine all into ppl_df
ppl_df = pd.concat([first_msg, days_active, total_msgs, type_counts, emoji_counts,
                    char_count, newline_count, punct_count, pct_upper], axis=1).reset_index()

print(ppl_df)


In [ ]:
# Define media types
media_types = ['audio', 'gif', 'photo', 'sticker', 'text', 'video']

# Total usage across all senders
media_usage_total = ppl_df[media_types].sum().sort_values(ascending=False)
print("Media usage (total per type):\n", media_usage_total)

# Optional: per sender breakdown
print(ppl_df[['sender'] + media_types])

# Factor to normalize per year
DAYS_IN_YEAR = 365

# Per-year normalization
ppl_df['msgs_per_year'] = ppl_df['total_messages'] / ppl_df['days_active'] * DAYS_IN_YEAR
ppl_df['emoji_per_year'] = ppl_df['n_emoji'] / ppl_df['days_active'] * DAYS_IN_YEAR
ppl_df['texts_per_year'] = ppl_df['text'] / ppl_df['days_active'] * DAYS_IN_YEAR
ppl_df['punct_per_year'] = ppl_df['n_punct'] / ppl_df['days_active'] * DAYS_IN_YEAR

# Media types per year
for mt in media_types:
    if mt in ppl_df.columns:
        ppl_df[f'{mt}_per_year'] = ppl_df[mt] / ppl_df['days_active'] * DAYS_IN_YEAR

# Display per-year stats
print(ppl_df[['sender', 'texts_per_year','msgs_per_year', 'emoji_per_year', 'punct_per_year'] + [f'{mt}_per_year' for mt in media_types]])

# Most/least messages
most_msgs = ppl_df.loc[ppl_df['msgs_per_year'].idxmax(), ['sender', 'msgs_per_year']]
least_msgs = ppl_df.loc[ppl_df['msgs_per_year'].idxmin(), ['sender', 'msgs_per_year']]

# Media usage per year
media_usage_per_year = ppl_df[[f'{mt}_per_year' for mt in media_types]].sum().sort_values(ascending=False)

# Most/least emoji usage
most_emoji = ppl_df.loc[ppl_df['emoji_per_year'].idxmax(), ['sender', 'emoji_per_year']]
least_emoji = ppl_df.loc[ppl_df['emoji_per_year'].idxmin(), ['sender', 'emoji_per_year']]

# Longest/shortest messages
df['msg_len'] = df['msg'].str.len()
longest_msg = df.loc[df['msg_len'].idxmax(), ['sender', 'msg_len', 'msg']]
shortest_msg = df.loc[df['msg_len'].idxmin(), ['sender', 'msg_len', 'msg']]

# Uppercase (shouting)
most_upper = ppl_df.loc[ppl_df['pct_uppercase'].idxmax(), ['sender', 'pct_uppercase']]
least_upper = ppl_df.loc[ppl_df['pct_uppercase'].idxmin(), ['sender', 'pct_uppercase']]

# Punctuation
most_punct = ppl_df.loc[ppl_df['punct_per_year'].idxmax(), ['sender', 'punct_per_year']]
least_punct = ppl_df.loc[ppl_df['punct_per_year'].idxmin(), ['sender', 'punct_per_year']]

# Print summary
print("Most messages/year:", most_msgs)
print("Least messages/year:", least_msgs)
print("Media usage per year (total per type):\n", media_usage_per_year)
print("Most emojis/year:", most_emoji)
print("Least emojis/year:", least_emoji)
print("Longest message:", longest_msg)
print("Shortest message:", shortest_msg)
print("Most shouting (% uppercase):", most_upper)
print("Least shouting (% uppercase):", least_upper)
print("Most punctuation/year:", most_punct)
print("Least punctuation/year:", least_punct)

In [ ]:
# filter down
text_df = df[df['type'] == 'text'].copy()

# helper: count emojis
def count_emoji(s: str) -> int:
    return sum(ch in emoji.EMOJI_DATA for ch in s)

text_df['emoji_count'] = text_df['msg'].apply(count_emoji)
text_df['char_count'] = text_df['msg'].str.len()

# aggregate
emoji_stats = (
    text_df.groupby('sender')[['emoji_count','char_count']]
    .sum()
    .assign(emoji_text_ratio=lambda x: x['emoji_count'] / x['char_count'].clip(lower=1))
    .sort_values('emoji_text_ratio', ascending=False)
)

emoji_stats.head(20)  # sender with highest ratio on top

In [ ]:
def uppercase_ratio(s: str) -> float:
    letters = re.findall(r'[A-Za-zÆØÅæøå]', s)
    if not letters:
        return 0.0
    upp = sum(ch.isupper() for ch in letters)
    return upp / len(letters)

text_df['pct_uppercase'] = text_df['msg'].apply(uppercase_ratio)

uppercase_stats = (
    text_df.groupby('sender')['pct_uppercase']
    .mean()
    .sort_values(ascending=False)
)

uppercase_stats.head(20)


In [ ]:
import re

# --- Load profanity list without pathlib ---
with open("profanity.txt", "r", encoding="utf-8") as f:
    profanity_words = [w.strip().lower() for w in f if w.strip()]

# build regex pattern (word-boundary safe)
pattern = re.compile(
    r'\b(' + '|'.join(map(re.escape, profanity_words)) + r')\b',
    re.IGNORECASE
)

# --- Counting functions ---
def count_profanity(text: str) -> int:
    if not isinstance(text, str):
        return 0
    return len(pattern.findall(text))

def word_count(s: str) -> int:
    return len(re.findall(r'\w+', s)) if isinstance(s, str) else 0

# --- Apply to DataFrame ---
df['profanity_count'] = df['msg'].apply(count_profanity)
df['word_count'] = df['msg'].apply(word_count)

# Only use text messages
text_df = df[df['type'] == 'text']

# Absolute profanity
profanity_stats = (
    text_df.groupby('sender')['profanity_count']
    .sum()
    .sort_values(ascending=False)
)

# Ratio (profanity / total words)
ratio_stats = (
    text_df.groupby('sender')
           .agg(
               total_profanity=('profanity_count', 'sum'),
               total_words=('word_count', 'sum')
           )
           .assign(profanity_ratio=lambda x: x['total_profanity'] / x['total_words'].clip(lower=1))
           .sort_values('profanity_ratio', ascending=False)
)

ratio_stats.head(20)

In [ ]:
# filter text messages only
text_df = df[df['type'] == 'text'].copy()

# regex pattern for words starting with "hah" (case-insensitive)
pattern = re.compile(r'\b(hah\w*)\b', re.IGNORECASE)

# extract all matches for each message
text_df['hah_words'] = text_df['msg'].apply(lambda x: pattern.findall(x) if isinstance(x, str) else [])

# explode so each word is one row
exploded = text_df.explode('hah_words').dropna(subset=['hah_words'])

# normalize to original case for breakdown
exploded['hah_words'] = exploded['hah_words']

# --- total count per person ---
total_counts = exploded.groupby('sender')['hah_words'].count().sort_values(ascending=False)
print("Total 'hah*' words per sender:")
print(total_counts)

# --- breakdown of variations per person ---
breakdown = exploded.groupby(['hah_words','sender']).size().unstack(fill_value=0)
print("\nBreakdown of variations per sender:")
print(breakdown)

# show all rows
pd.set_option('display.max_rows', None)

# show all columns
pd.set_option('display.max_columns', None)

# optionally, show full column width (no truncation)
pd.set_option('display.max_colwidth', None)

# get all unique variations
unique_hah = exploded['hah_words'].dropna().unique()

# sort by length
unique_hah_sorted = sorted(unique_hah, key=len)

print("Unique 'hah*' variations sorted by length:")
for word in unique_hah_sorted:
    print(word)


pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.max_colwidth')

In [ ]:
# compute length of each hah word
exploded['hah_length'] = exploded['hah_words'].str.len()

# average length per sender
avg_hah_length = exploded.groupby('sender')['hah_length'].mean().sort_values(ascending=False)

print("Average 'hah*' length per sender:")
print(avg_hah_length)
